# Week 08 — BBO capstone driver

Round 8. Perturbations of magnitude ≤0.001 around the W7 points — the third consecutive round in the same neighbourhood and the smallest step yet.

**This is the low point of the campaign's budget management**, and it is also where the campaign turns. F8's return moves from 8.6894 to 8.7507 across an input distance of ~0.057, giving a directional derivative of about 1.08 per unit distance. That is a real finite difference and a signed direction, obtained by accident.

F3 also returns -0.015429 here, which stands as its best for the rest of the project.

From this round onward the proposals stop being points and start being **directions**. The cost of learning that was three rounds.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 8
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 8
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: '≤0.001 perturbation of W7',
    2: '≤0.001 perturbation of W7',
    3: '≤0.001 perturbation of W7',
    4: '≤0.001 perturbation of W7',
    5: '≤0.001 perturbation of W7',
    6: '≤0.001 perturbation of W7',
    7: '≤0.001 perturbation of W7',
    8: '≤0.001 perturbation of W7',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 7. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — ≤0.001 around W7, and the finite difference that falls out

`bbo.finite_difference` turns a pair of nearby queries into a unit direction and a directional derivative.

In [ ]:
proposals = {
    1: np.array([0.669937, 0.751452]),
    2: np.array([0.354432, 0.449899]),
    3: np.array([0.111275, 0.774316, 0.516951]),
    4: np.array([0.183872, 0.065353, 0.017618, 0.904326]),
    5: np.array([0.308806, 0.730874, 0.091998, 0.661917]),
    6: np.array([0.070021, 0.530732, 0.952853, 0.825805, 0.228797]),
    7: np.array([0.965568, 0.153915, 0.588691, 0.807159, 0.099427, 0.700138]),
    8: np.array([0.038733, 0.275735, 0.181777, 0.348003, 0.803094, 0.234913, 0.935841, 0.094663]),
}

# The F8 pair W6 -> W8 is far enough apart to carry a real gradient.
x_a = np.array(bbo.HISTORY[6][8][0]); y_a = bbo.HISTORY[6][8][1]
x_b = np.array(bbo.HISTORY[8][8][0]); y_b = 8.7506992510951  # this round's return
direction, deriv = bbo.finite_difference(x_a, y_a, x_b, y_b)
print(f"F8 directional derivative: {deriv:+.4f} per unit distance")
print(f"F8 unit direction:         {np.round(direction, 4)}")
print(f"distance between the pair: {np.linalg.norm(x_b - x_a):.4f}")


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.669237, 0.752052],
    2: [0.355232, 0.449399],
    3: [0.110375, 0.775016, 0.516551],
    4: [0.184472, 0.064553, 0.017918, 0.903826],
    5: [0.308106, 0.731274, 0.091098, 0.662517],
    6: [0.069721, 0.531532, 0.952453, 0.826305, 0.228097],
    7: [0.965268, 0.154515, 0.588291, 0.807859, 0.098927, 0.700438],
    8: [0.018683, 0.255785, 0.161747, 0.328043, 0.783074, 0.214943, 0.955801, 0.074683],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 8 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: -7.058138619523124e-14,
#     2: 0.0836536493073266,
#     3: -0.01542931285755213,
#     4: -23.621861521751097,
#     5: 0.5222091523300119,
#     6: -0.9676493116712739,
#     7: 0.07982491545502426,
#     8: 8.7506992510951,
# }
#
# F3 -0.015429: best for the remainder of the project.
# F8 8.7507 from 8.6894: the accidental finite difference that reframes every
# subsequent round. Note also F5 at 0.522 - the incumbent the next five rounds
# anchor on, and roughly 1/1000th of what W1 already returned.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
